# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Keywords: {', '.join(metadata.keywords) if hasattr(metadata, 'keywords') else 'N/A'}")

## 2. Data Overview
Review available record sets, their fields, and their IDs.

**Note:** All entities are referenced by their `@id`.

In [ ]:
# List all available record sets and their fields by @id
from pprint import pprint

print("Record Sets: (referenced by @id)")
record_set_ids = []
for record_set in dataset.record_sets:
    print(f"- Record Set: {record_set.id}")
    record_set_ids.append(record_set.id)
    for field in record_set.fields:
        print(f"   - Field: {field.id} (type: {getattr(field, 'data_type', 'N/A')})")

# For demonstration, print a sample record for each record set
for record_set in dataset.record_sets:
    print(f"\nSample record from {record_set.id}: ")
    records = dataset.records(record_set=record_set.id)
    for idx, rec in enumerate(records):
        pprint(rec)
        if idx > 0:
            break

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. All names are referenced by their `@id`.

We'll demonstrate this for all record sets found above.

In [ ]:
# Extract data from each record set by @id
dataframes = {}
for record_set in record_set_ids:
    print(f"Loading records for Record Set @id: {record_set}")
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df
    print(f"Columns for {record_set}:")
    print(df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field from the first record set above for simple analysis. 

Typical protein MS datasets include fields like `cr:coverage` (percent coverage), `cr:peptide_count`, or `cr:abundance` (could vary depending on the schema). For demonstration, we search for a likely numeric field to use.

In [ ]:
# Select a numeric field and perform EDA (using the first record set as example)
record_set_id = record_set_ids[0]
df = dataframes[record_set_id].copy()

# Try to automatically detect a numeric field
numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
if not numeric_candidates:
    possible_numeric = [c for c in df.columns if any(substr in c.lower() for substr in ['coverage', 'count', 'abundance', 'weight', 'mw', 'pi', 'mz', 'score'])]
    print(f"No direct numeric columns found. Candidates based on name: {possible_numeric}")
    numeric_field = possible_numeric[0] if possible_numeric else df.columns[0]
    # Attempt conversion
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
else:
    numeric_field = numeric_candidates[0]

print(f"Selected numeric field: {numeric_field}")

# Filtering records
threshold = 10
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping if possible
possible_group_fields = [c for c in df.columns if 'group' in c.lower() or 'category' in c.lower() or 'class' in c.lower() or 'sample' in c.lower()]
group_field = possible_group_fields[0] if possible_group_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped data by {group_field} (mean of {numeric_field}):")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution and relationship of the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.show()

# If we have grouped data, show a boxplot
if group_field:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.xticks(rotation=45)
    plt.title(f'{numeric_field} by {group_field}')
    plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load, explore, and analyze the FAIR² dataset: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells.

- We listed the record sets and fields using their `@id`s.
- We loaded sample data into Pandas DataFrames.
- Using EDA, we filtered, normalized, and visualized a selected quantitative field.
- This process can be adapted for any Croissant dataset by changing the schema URL at the start.

**Next Steps:** Deeper domain analysis can be performed by leveraging the field semantics and external metadata described in the schema, for more specific applications such as proteomics machine learning tasks or biomarker discovery.